# 08 — Assembling the S&P 500 Item 1 table

**Purpose.** Turn 21,867 extracted text files plus 432 gaps into one clean table:
one row per S&P 500 annual report, carrying the Item 1 (Business) text, where it
came from, and whether it is usable.

Three things happen here, in order, and each is measured rather than assumed.

## 1. Fill the gaps the API left

The Extractor returned nothing for **432 of 22,299 filings (1.94 percent)**. Item
1 is mandatory in a 10-K, so an empty result is a failure by definition — and the
text is there: FactSet, Marathon Oil, Iron Mountain and E*TRADE all have full
Business sections the API declines to return. It is deterministic, not transient:
four URL shapes and both output modes were tried on eleven of them and all
twenty-two attempts returned zero characters.

`lib/item1_local.py` reads the archived document instead. Against the API on
5,314 filings where the API succeeded it agrees at a **median token Jaccard of
0.9976**, with 95.6 percent at or above 0.95.

## 2. Clean the text for use as LLM input

This text is fed to a model directly, so the cleaning target is not tidiness —
it is that **nothing foreign sits inside a sentence**. Two artefacts break that,
and they are not cosmetic:

* **`##PAGE` and `<PAGE>` sentinels**, in 20.5 percent of files, 47,961 of them.
  They mark a page break in the original document, which lands wherever the page
  happened to end — usually mid-sentence:

  > ...fiber optic video transmission equipment for the telephone, cable
  > television, broadcast and government `##PAGE` markets.

  The sentence is "broadcast and government markets". A model reading it sees a
  markdown heading wedged between "government" and "markets".

* **Bare page numbers**, in 27.3 percent of files, immediately beside those
  sentinels and splitting words the same way: "...voice, data and video
  networking `2` `##PAGE` capabilities within private networks...".

* **Hard wrapping.** 48.1 percent of documents are wrapped to a fixed width —
  98.6 percent of 1990s filings, at a median line length of 74 characters — which
  puts a newline inside a sentence every few words. 2.1 million of them. Joining
  those lines back costs nothing and removes the fragmentation.

Also dropped: `##TABLE_START`/`##TABLE_END` (67.4 percent), HTML entities (76.2
percent, unescaped **twice** because some are double-encoded), dot leaders and
rule characters used for column alignment.

**Deliberately kept.** The section glyph `§` is a legal citation — "Clean Air Act
§7604" — not decoration. Financial tables are content: the dot runs in "First
mortgage bonds........ $1,729,070" are alignment padding and collapse to a space,
but the line stays.

## 3. Cut the contamination

23.5 percent of files contain the string "Item 1A". **Almost all of that is
legitimate**, and the naive fix would destroy real text: the phrase usually
appears inside a sentence —

> ...factors that could cause our actual results to differ materially are
> described under "**Item 1A. Risk Factors**" below. You are cautioned not to...

Position does not separate the two cases: those cross-references sit at 1 percent
of the file with the entire Business section after them, which is exactly what a
naive "big tail after the match" rule would flag as contamination.

What does separate them is whether the mention **starts a line**. On that test
6,297 files carry only cross-references and **110 carry a real heading**, and the
110 split into two kinds that need opposite treatment:

* **A leading table of contents** — the heading sits at 0-1 percent of the file
  and is followed by more item headings (`Item 1A ... Item 1B ... Item 2 ...
  Part II`). The Business text comes after. Cut the front.
* **A genuine Item 1A section** — the heading sits mid-document and is followed
  by risk-factor prose ("When evaluating our company... consider the following
  risk factors"). Cut the back.

**Inputs**

| Source | Path |
|---|---|
| Extracted text | `interim/sec/sp500_item1/item1/{cik}/{accessionNo}.txt` |
| Status log | `interim/sec/sp500_item1/status/extract_*.jsonl` |
| Filing index | `raw/sec/filings.parquet` — from `08` |

**Output**

`data/raw/03_sp500_10k_item1/sp500_10k_item1.parquet`, zstd. About 370 MB, which
is past GitHub's 100 MB limit, so the directory is gitignored — the file is a
local research artifact this notebook regenerates, not repository content.

## Setup

In [1]:
import glob
import html
import json
import re
import sys
from pathlib import Path

sys.path.insert(0, "../lib")

import pandas as pd

from item1_local import extract_item1
from paths import DATA_REPO, SEC_INTERIM, SEC_RAW

RUN_DIR = SEC_INTERIM / "sp500_item1"
OUT_DIR = DATA_REPO / "raw" / "03_sp500_10k_item1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Below this a "section" is a pointer, not a business description. Set from the
# measured distribution: 216 API results fall under it and every one inspected
# was an incorporation by reference ("Pages 20 through 28 of the Annual Report").
MIN_USABLE_CHARS = 1000

print(f"run    : {RUN_DIR}")
print(f"output : {OUT_DIR}")

run    : /scratch2/acroh/projects/llm_method/data/interim/sec/sp500_item1
output : /project2/njia_1089/acroh/projects/llm_method/data/raw/03_sp500_10k_item1


## Step 1 — Load the status log and the filing index

The log is the authority on what was attempted; the index supplies the firm
identifiers. Joining them here means the table carries GVKEY and fiscal year
without any later lookup.

In [2]:
rows = [json.loads(line)
        for p in sorted(glob.glob(str(RUN_DIR / "status" / "extract_*.jsonl")))
        for line in open(p) if line.strip()]
status = pd.DataFrame(rows).drop_duplicates(["accessionNo", "item"], keep="last")

idx = pd.read_parquet(SEC_RAW / "filings.parquet")
sp = idx.loc[idx["in_sp500"]].copy()

assert len(status) == len(sp), (
    f"{len(status):,} status rows against {len(sp):,} S&P 500 filings — "
    "the extraction did not cover the index"
)

print(f"status rows : {len(status):,}")
print(status["status"].value_counts().to_string())
print(f"\nS&P 500 filings in the index: {len(sp):,}")

status rows : 22,299
status
ok       21867
empty      432

S&P 500 filings in the index: 22,299


## Step 2 — Recover the filings the API left empty

`extract_item1` returns its own status, and the three outcomes are kept apart
because they mean different things. `ok` is recovered text. `incorporated_by
_reference` means the filing really does only point elsewhere — not a failure,
and not usable text either. `not_found` means no Item 1 heading could be located,
which is the honest answer for a filing whose structure neither the API nor this
module can parse.

In [3]:
empty = status.loc[status["status"] == "empty", "accessionNo"].tolist()
container = idx.set_index("accessionNo")["container"]

recovered = {}
counts = {"ok": 0, "incorporated_by_reference": 0, "not_found": 0, "error": 0}
for acc in empty:
    try:
        r = extract_item1(acc, container.get(acc))
    except Exception as exc:                      # noqa: BLE001 - reported, not raised
        counts["error"] += 1
        recovered[acc] = {"status": "error", "text": "", "note": f"{type(exc).__name__}: {exc}"}
        continue
    counts[r.get("status", "not_found")] = counts.get(r.get("status", "not_found"), 0) + 1
    recovered[acc] = r

print(f"attempted recovery on {len(empty):,} filings the API returned nothing for")
for key, value in counts.items():
    if value:
        print(f"  {key:<26} {value:>5,}")
got = [len(r.get("text") or "") for r in recovered.values() if r.get("status") == "ok"]
if got:
    s = pd.Series(got)
    print(f"\nrecovered text: median {s.median():,.0f} chars, max {s.max():,}")

attempted recovery on 432 filings the API returned nothing for
  ok                           334
  incorporated_by_reference     29
  not_found                     69

recovered text: median 44,856 chars, max 252,461


## Step 3 — Assemble one row per filing

`source` records where each row's text came from, so a later analysis can drop
the locally extracted rows if it wants API-only provenance.

In [4]:
def read_api_text(path):
    try:
        return Path(path).read_text()
    except (OSError, TypeError):
        return ""


records = []
for row in status.itertuples():
    if row.status == "ok":
        text, source, note = read_api_text(row.path), "sec-api", ""
    elif row.accessionNo in recovered:
        r = recovered[row.accessionNo]
        text = r.get("text") or ""
        source = "local" if r.get("status") == "ok" else "none"
        note = r.get("status", "")
    else:
        text, source, note = "", "none", row.status
    records.append({"accessionNo": row.accessionNo, "text_raw": text,
                    "source": source, "recovery_note": note})

items = pd.DataFrame(records)
keep = ["accessionNo", "cik_unpadded", "gvkey", "conm", "companyName", "formType",
        "fyear", "periodEnd", "filedAt", "filingUrl", "url_bare"]
items = items.merge(sp[keep], on="accessionNo", how="left", validate="one_to_one")

print(items["source"].value_counts().to_string())
print(f"\n{len(items):,} rows")

source
sec-api    21867
local        334
none          98

22,299 rows


## Step 4 — Clean the text

Order matters here. Page sentinels and page numbers collapse to a single newline
first; only then are wrapped lines rejoined, so a sentence broken by a page break
is put back together by the same rule that undoes hard wrapping.

A line is treated as wrapped when it does not end in sentence-final punctuation
**and** the next line starts lowercase. That pair of conditions is what keeps
headings, list items and table rows on their own lines while rejoining prose.

In [5]:
TABLE_MARK = re.compile(r"##TABLE_(?:START|END)")
UNI_SPACE = re.compile("[   -   　]")
ZERO_WIDTH = re.compile("[​‌‍⁠﻿­]")

# Page-break sentinels, in both the API's form and the original SGML's.
PAGE_MARK = re.compile(r"(?i)[ \t]*(?:##[ \t]*PAGE|<PAGE>)[ \t]*")
# A line holding only a page number, with or without decoration: "12", "- 12 -".
PAGE_NUM = re.compile(r"\n[ \t]*-{0,2}[ \t]*\d{1,4}[ \t]*-{0,2}[ \t]*(?=\n)")
# Alignment fill: dot leaders and rule characters. The surrounding text stays.
LEADER = re.compile(r"[.\u00b7]{4,}|_{3,}|-{4,}|={3,}|\*{3,}")
# A newline that is inside a sentence: the line does not end and the next
# continues in lower case. Anything else — a heading, a list item, a table row —
# fails one of the two conditions and keeps its break.
WRAP = re.compile(r"(?<![.:;!?\)\]\"\'])\n(?=[a-z])")


def clean_text(t: str) -> str:
    if not t:
        return ""
    t = html.unescape(html.unescape(t))     # twice: some output is double-encoded
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = ZERO_WIDTH.sub("", t)
    t = UNI_SPACE.sub(" ", t)
    t = TABLE_MARK.sub(" ", t)

    # Page furniture first, down to a single newline, so the unwrap below can
    # rejoin the sentence the page break split.
    t = PAGE_MARK.sub("\n", t)
    t = PAGE_NUM.sub("\n", t)
    t = LEADER.sub(" ", t)

    t = re.sub(r"[ \t]{2,}", " ", t)
    t = re.sub(r"[ \t]*\n[ \t]*", "\n", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    t = WRAP.sub(" ", t)                    # undo hard wrapping and page splits

    t = re.sub(r"[ \t]{2,}", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


before = items["text_raw"].str.len()
items["text_clean"] = items["text_raw"].map(clean_text)
after = items["text_clean"].str.len()

alive = before > 0
print(f"cleaning removed {(before - after)[alive].sum() / 1e6:.1f}M characters "
      f"({(1 - after[alive].sum() / before[alive].sum()):.1%} of the corpus)")
print(f"  median shrink per document: {((1 - after[alive] / before[alive])).median():.1%}")
for name, pat in [("HTML entities", r"&#\d+;|&[a-z]{2,8};"),
                  ("table markers", r"##TABLE_"),
                  ("page sentinels", r"(?i)##\s*PAGE|<PAGE>"),
                  ("bare page numbers", r"\n[ \t]*\d{1,4}[ \t]*\n"),
                  ("dot leaders", r"\.{4,}"),
                  ("mid-sentence newline", r"(?<![.:;!?\)\]])\n(?=[a-z])"),
                  ("4+ space runs", r"[ \t]{4,}"),
                  ("4+ newline runs", r"\n{4,}")]:
    b = items["text_raw"].str.contains(pat, regex=True, na=False).sum()
    a = items["text_clean"].str.contains(pat, regex=True, na=False).sum()
    print(f"  {name:<16} {b:>7,} -> {a:>6,}")

cleaning removed 74.8M characters (6.6% of the corpus)
  median shrink per document: 4.3%


  HTML entities     16,665 ->      0


  table markers     14,731 ->      0


  page sentinels     4,497 ->      0


  bare page numbers   4,351 ->     17


  dot leaders        1,377 ->      0


  mid-sentence newline  17,660 ->  2,658


  4+ space runs      5,038 ->      0


  4+ newline runs    2,716 ->      0


## Step 5 — Find the item headings that start a line

This is the test that separates a heading from a cross-reference, and it is the
whole reason the cut can be made safely. A cross-reference has sentence text to
its left on the same line; a heading does not.

Both `Item 1A` and `Item 2`/`Item 3` are looked for, because a filing that
predates Item 1A ends at Properties or Legal Proceedings instead.

In [6]:
HEADING = re.compile(
    r"(?:^|\n)[ \t]*(?:part\s+i[ \t]*[\-–—]?[ \t]*)?"
    r"item[ \t]*(1A|1B|2|3)[ \t]*[\.\:\-–—]?[ \t]*"
    r"(risk[ \t]*factors?|unresolved|propert|legal)",
    re.I)


def line_initial_headings(t: str):
    """Every item heading that begins its own line, as (position, item)."""
    out = []
    for mt in HEADING.finditer(t):
        line_start = t.rfind("\n", 0, mt.start() + 1) + 1
        # Anything other than whitespace to the left on this line means the
        # phrase is inside a sentence — a cross-reference, not a heading.
        if t[line_start:mt.start()].strip():
            continue
        out.append((mt.start(), mt.group(1).upper()))
    return out


items["headings"] = items["text_clean"].map(line_initial_headings)
has = items["headings"].map(len) > 0
xref = items["text_clean"].str.contains(r"item\s*1A", case=False, regex=True, na=False) & ~has

print(f"line-initial item headings found in : {int(has.sum()):>6,} documents")
print(f"'Item 1A' present only as prose      : {int(xref.sum()):>6,} documents")
print(f"neither                              : {int((~has & ~xref).sum()):>6,} documents")

line-initial item headings found in :    211 documents
'Item 1A' present only as prose      :  6,301 documents
neither                              : 15,787 documents


## Step 6 — Cut the front, or cut the back

A heading in the first tenth of the document, with another heading close behind
it, is a table of contents: the run of `Item 1A ... Item 1B ... Item 2 ... Part
II` that some filings carry before the Business text. Everything up to the last
of those goes.

A heading anywhere else is the start of the next section, and everything from it
goes instead — with no minimum on how much follows. A heading at 99 percent of
the document is a stray boundary marker (`\nItem 1A. Risk Factors` and nothing
after it) and a heading at 20 percent is a swallowed section; both should go.

Both cuts are recorded per row so the effect is auditable rather than silent.

In [7]:
TOC_ZONE = 0.10          # a heading this early is part of a front table of contents
TOC_GAP = 400            # consecutive TOC rows sit close together
MIN_TAIL = 2000          # only guards the TOC rule; the tail cut has no minimum


def cut(row):
    t, hs = row["text_clean"], row["headings"]
    if not t or not hs:
        return pd.Series([t, "none", 0])

    n = len(t)
    front = [p for p, _ in hs if p < n * TOC_ZONE]
    # A table of contents is a RUN of headings, not one heading. A single early
    # heading is more likely a real short Item 1 than a contents block.
    if len(front) >= 2 and max(b - a for a, b in zip(front, front[1:])) < TOC_GAP:
        end = max(front)
        end = t.find("\n", end + 40)
        if end > 0 and (n - end) > MIN_TAIL:
            return pd.Series([t[end:].strip(), "leading_toc", end])

    # No minimum on the tail. A heading at 99 percent of the document is a stray
    # boundary marker — "\nItem 1A. Risk Factors" and nothing after it — and a
    # heading at 20 percent is a swallowed section. Both should go, and requiring
    # a minimum tail only preserves the first kind for no reason.
    tail = [p for p, _ in hs if p >= n * TOC_ZONE]
    if tail:
        return pd.Series([t[:min(tail)].strip(), "next_section", n - min(tail)])
    return pd.Series([t, "none", 0])


items[["item1", "cut_kind", "cut_chars"]] = items.apply(cut, axis=1)

print(items["cut_kind"].value_counts().to_string())
c = items.loc[items["cut_kind"] != "none"]
if len(c):
    print(f"\n{len(c):,} documents cut, {c['cut_chars'].sum() / 1e6:.2f}M characters removed")
    print(c.groupby("cut_kind")["cut_chars"].describe()[["count", "50%", "max"]].to_string())

cut_kind
none            22089
next_section      205
leading_toc         5

210 documents cut, 3.30M characters removed
              count     50%       max
cut_kind                             
leading_toc     5.0   172.0     762.0
next_section  205.0  1399.0  271248.0


## Step 7 — Label every row

`usable` is the column an analysis should filter on. It is deliberately strict:
text has to exist, be long enough to be a description rather than a pointer, and
not be a leftover contents block.

In [8]:
items["chars"] = items["item1"].str.len()
items["incorporated_by_reference"] = (
    (items["chars"] > 0) & (items["chars"] < MIN_USABLE_CHARS)
)
items["starts_at_item1"] = items["item1"].str.match(
    r"\W{0,80}(part\s+i\b.{0,40})?item[s]?\s*1\b", case=False, na=False)
items["usable"] = items["chars"] >= MIN_USABLE_CHARS

print(f"usable                      {int(items['usable'].sum()):>7,}  "
      f"({items['usable'].mean():.2%})")
print(f"incorporated by reference   {int(items['incorporated_by_reference'].sum()):>7,}")
print(f"no text at all              {int((items['chars'] == 0).sum()):>7,}")
print()
print(items.groupby("source")["usable"].agg(["size", "sum", "mean"]).to_string())

usable                       21,966  (98.51%)
incorporated by reference       264
no text at all                   69

          size    sum      mean
source                         
local      334    334  1.000000
none        98      0  0.000000
sec-api  21867  21632  0.989253


## Step 8 — Write the table

In [9]:
COLUMNS = ["accessionNo", "cik_unpadded", "gvkey", "conm", "companyName", "formType",
           "fyear", "periodEnd", "filedAt", "filingUrl", "source", "recovery_note",
           "cut_kind", "cut_chars", "chars", "starts_at_item1",
           "incorporated_by_reference", "usable", "item1"]

out = items[COLUMNS].rename(columns={"cik_unpadded": "cik"}).sort_values(["gvkey", "fyear"])
target = OUT_DIR / "sp500_10k_item1.parquet"
out.to_parquet(target, compression="zstd", index=False)

size = target.stat().st_size
print(f"wrote {target}")
print(f"  {len(out):,} rows, {size / 1e6:.0f} MB zstd")
print(f"  {out['item1'].str.len().sum() / 1e9:.2f} billion characters of text")
print(f"  {out['gvkey'].nunique():,} firms, fiscal years "
      f"{int(out['fyear'].min())}-{int(out['fyear'].max())}")

wrote /project2/njia_1089/acroh/projects/llm_method/data/raw/03_sp500_10k_item1/sp500_10k_item1.parquet
  22,299 rows, 145 MB zstd
  1.06 billion characters of text
  1,061 firms, fiscal years 1992-2026


## Verification

In [10]:
back = pd.read_parquet(target)

assert len(back) == len(sp), f"{len(back):,} rows against {len(sp):,} S&P 500 filings"
print(f"1) one row per S&P 500 filing ({len(back):,})")

assert not back["accessionNo"].duplicated().any(), "duplicate accessionNo"
assert back["gvkey"].notna().all(), "rows without a GVKEY"
print("2) accession numbers unique, every row carries a GVKEY")

leftover = back.loc[back["usable"], "item1"].str.contains(
    r"##TABLE_|##\s*PAGE|<PAGE>|&#\d+;", regex=True, case=False, na=False).sum()
assert leftover == 0, f"{leftover} usable rows still carry markup artefacts"
print("3) no table/page markers or HTML entities left in usable text")

frag = back.loc[back["usable"], "item1"].str.count(r"(?<![.:;!?\)\]])\n(?=[a-z])").sum()
print(f"   mid-sentence newlines remaining: {int(frag):,}")

# The section glyph is a legal citation, not decoration, and must survive.
kept = back.loc[back["usable"], "item1"].str.contains("§", na=False).sum()
print(f"   documents still carrying a § legal citation: {int(kept):,}")

# The cut must not have eaten the document. Anything that survives as usable
# should still begin at Item 1 far more often than not.
rate = back.loc[back["usable"], "starts_at_item1"].mean()
print(f"4) usable text begins at an Item 1 heading in {rate:.1%} of rows")

still = back.loc[back["usable"], "item1"].map(
    lambda t: bool(re.search(r"(?:^|\n)[ \t]*item[ \t]*1A[ \t]*[\.\:]?[ \t]*risk",
                             t, re.I)))
print(f"5) usable rows still holding a line-initial 'Item 1A. Risk Factors': "
      f"{int(still.sum()):,}")

print()
print("6) coverage by decade:")
cov = back.assign(decade=back["fyear"] // 10 * 10).groupby("decade").agg(
    filings=("accessionNo", "size"), usable=("usable", "sum"),
    median_chars=("chars", "median"))
cov["share"] = (cov["usable"] / cov["filings"]).map("{:.1%}".format)
print(cov.to_string())

1) one row per S&P 500 filing (22,299)
2) accession numbers unique, every row carries a GVKEY


3) no table/page markers or HTML entities left in usable text


   mid-sentence newlines remaining: 5,337


   documents still carrying a § legal citation: 166
4) usable text begins at an Item 1 heading in 98.0% of rows


5) usable rows still holding a line-initial 'Item 1A. Risk Factors': 0

6) coverage by decade:
        filings  usable  median_chars  share
decade                                      
1990       3987    3848       29437.0  96.5%
2000       7351    7255       39135.0  98.7%
2010       7012    6969       41372.5  99.4%
2020       3949    3894       46486.0  98.6%
